# SMS Spam Detection using Naive Bayes

**Repositori**: Machine Learning  
**Topik**: Implementasi Naive Bayes pada Klasifikasi SMS Spam  
**Dataset**: `SMSSpam.csv`  

---

## Pendahuluan
Proyek ini mendemonstrasikan implementasi **Multinomial Naive Bayes** untuk tugas klasifikasi teks (Spam vs Ham). Naive Bayes sangat cocok untuk klasifikasi teks karena bekerja dengan baik pada data sparse hasil TF-IDF dan memberikan output probabilitas secara alami.

### Alur Kerja (Pipeline):
1. **Data Acquisition**: Mengambil data dari CSV.
2. **EDA (Exploratory Data Analysis)**: Menganalisis distribusi dan karakteristik data.
3. **Data Preparation**: Cleaning dan Label Encoding (Ham=0, Spam=1).
4. **Feature Engineering**: Transformasi teks menggunakan TF-IDF.
5. **Modeling**: Training menggunakan Multinomial Naive Bayes.
6. **Evaluation**: Menggunakan Confusion Matrix, Accuracy, Precision, dan Recall.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_curve, auc

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Acquisition & Understanding
Memuat dataset dan melihat struktur data awal.

In [ ]:
df = pd.read_csv('../../data/SMSSpam.csv', names=['Label', 'Message'], encoding='latin-1')

print("Shape Dataset:", df.shape)
df.head()

## 2. Exploratory Data Analysis (EDA)
Menganalisis karakteristik dataset sebelum preprocessing.

In [ ]:
print("Missing Values:\n", df.isnull().sum())

print("\nDistribusi Label:")
print(df['Label'].value_counts())
print(f"\nPersentase:\n{df['Label'].value_counts(normalize=True).mul(100).round(2)}")

df['Message_Length'] = df['Message'].apply(len)
print(f"\nStatistik Panjang Pesan:")
print(df['Message_Length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='Message_Length', hue='Label', bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Panjang Pesan')
sns.boxplot(data=df, x='Label', y='Message_Length', palette='viridis', ax=axes[1])
axes[1].set_title('Boxplot Panjang Pesan per Label')
plt.tight_layout()
plt.show()

## 3. Data Preparation & Cleaning
Melakukan encoding pada label dan membersihkan data jika diperlukan.

In [ ]:
df['Label_Num'] = df['Label'].map({'ham': 0, 'spam': 1})

sns.countplot(x='Label', data=df, hue='Label', palette='viridis', legend=False)
plt.title('Distribusi Ham vs Spam')
plt.show()

## 4. Feature Engineering (TF-IDF)
Mengubah teks pesan menjadi representasi angka menggunakan TF-IDF Vectorizer.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
X = tfidf.fit_transform(df['Message'])
y = df['Label_Num']

# Split data: Training 60%, Validation 20%, Testing 20% (sama seperti Linear Regression)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Data Training: {X_train.shape[0]} sampel")
print(f"Data Validation: {X_val.shape[0]} sampel")
print(f"Data Testing: {X_test.shape[0]} sampel")

## 5. Model Training (Multinomial Naive Bayes)
Melatih model dan mengevaluasi performa pada validation set.

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

# Prediksi probabilitas pada validation set untuk ROC Curve
y_val_proba = model.predict_proba(X_val)[:, 1]

# ROC Curve validation set
fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc(fpr, tpr):.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red',
            label=f'Optimal Threshold = {optimal_threshold:.3f}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Validation Set (Naive Bayes)')
plt.legend()
plt.show()

print(f"Threshold Optimal: {optimal_threshold:.3f}")
print(f"TPR (Recall): {tpr[optimal_idx]:.3f}")
print(f"FPR: {fpr[optimal_idx]:.3f}")

# Prediksi pada test set
y_test_proba = model.predict_proba(X_test)[:, 1]
y_pred = [1 if x >= optimal_threshold else 0 for x in y_test_proba]

## 6. Performance Evaluation
Mengevaluasi model menggunakan Confusion Matrix, Accuracy, Precision, Recall, dan F1-Score.

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Naive Bayes')
plt.show()

print("--- Laporan Klasifikasi ---")
print(classification_report(y_test, y_pred))

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2f}")

## 7. Perbandingan dengan Linear Regression

| Metrik | Linear Regression | Naive Bayes |
|--------|:-:|:-:|
| Accuracy | 0.88 | ? |
| Precision (Spam) | 0.55 | ? |
| Recall (Spam) | 0.75 | ? |
| F1-Score (Spam) | 0.63 | ? |
| AUC | ? | ? |

Catatan: Nilai Linear Regression diambil dari notebook `Linear-Regression-SMS.ipynb`. Setelah menjalankan notebook ini, isi tabel perbandingan di atas untuk melihat model mana yang lebih baik untuk klasifikasi SMS Spam.

## Kesimpulan
Multinomial Naive Bayes umumnya lebih unggul daripada Linear Regression untuk klasifikasi teks karena:
1. Naive Bayes memodelkan probabilitas secara alami, tidak perlu threshold tuning.
2. Cocok untuk data sparse hasil TF-IDF.
3. Lebih efisien dan interpretable untuk teks classification.

Jalankan semua sel untuk melihat hasilnya dan bandingkan dengan Linear Regression!